In [ ]:
%load_ext autoreload
%autoreload 2
import torch
from dotenv import load_dotenv
from accelerate import Accelerator
from constant import *
from GeminiModel import GeminiModel
from TrainStrategy import TrainStrategy

In [ ]:
load_dotenv()
accelerator = Accelerator()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
DETECTION_TEMPLATE = PromptTemplate(
    name="Manually Crafted",
    definition="You are Code Expert trained to detect Self-Admitted Technical Debt (SATD) in Java test code comments. SATD occurs when developers explicitly acknowledge that the current implementation is suboptimal, requires improvement, or contains technical compromises. These comments often include markers (e.g., TODO, FIXME), indicate unresolved issues, temporary fixes (e.g., workarounds, hacks), performance concerns, deprecated API usage, unsupported features, poor design, skipped tests, or unknown reasons. However, do not classify comments that merely describe expected behavior, actions, instructions or simply issue references, unless there is additional information indicating the need for future improvement. These comments often appear as imperative sentence structures (e.g., check argument, should not match), vague single-word description(e.g., clean up, retry, fail).",
    instruction="Classify by labelling it as 'yes' if the comment include a strong indication of Self-Admitted Technical Debt otherwise label it as 'no', do not return reason. Do not provide a reason for the classification.",
    n_shot_template="""
    <EXAMPLE>
    Comment: {{ text }}
    {% if cot -%}
    Reason: {{ cot }}
    {% endif -%}
    Label: {{ label }}
    </EXAMPLE>""",
    line_m_before=3,
    line_n_after=3
)

# Detect with Gemini 2.0 Flash N-Shots

In [ ]:
gemini_2_flash_detection_model = GeminiModel('detect', 'models/gemini-2.0-flash', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
gemini_2_flash_detection_model.fit(detect_n_shot_dataset)
gemini_2_flash_detection_model.predict(detect_test_dataset, DETECTION_TEMPLATE, TrainStrategy.N_SHOT_TOP, 10, verbose=False)